# 0. Synthetic Dummy Dataset Generator (`./data/`)

Author: Bram Schonfeldt

This notebook generates synthetic datasets in `./data/internal/` and `./data/public/` that match the exact schema, MultiIndex dimensions, and checkpoint hashes expected by the evaluation notebooks (`1_demographic_pyramids.ipynb` through `5_uk_trends.ipynb`).

Run all cells in this notebook once before running notebooks `1`–`5` in an external environment.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import pickle as pkl
from typing import Any
import numpy as np
import pandas as pd

# Resolve the repository root ./data directory whether run from repo root or colab/analysis/
REPO_ROOT = (
    os.path.abspath('../..')
    if os.path.exists(os.path.join('..', '..', 'stratify.py'))
    else os.path.abspath('.')
)
DATA_DIR = os.path.join(REPO_ROOT, 'data')
print(f'Target data directory: {DATA_DIR}')

In [ ]:
AGE_BRACKETS = [
    '20-24',
    '25-29',
    '30-34',
    '35-39',
    '40-44',
    '45-49',
    '50-54',
    '55-59',
    '60-64',
    '65-69',
    '70-74',
    '75-79',
    '80-84',
    '85-89',
    '90-94',
    '95-99',
    '100+',
]
MIDAGES = [
    22.5,
    27.5,
    32.5,
    37.5,
    42.5,
    47.5,
    52.5,
    57.5,
    62.5,
    67.5,
    72.5,
    77.5,
    82.5,
    87.5,
    92.5,
    97.5,
    102.5,
]
SAMPLE_COUNTRIES = [
    'Japan',
    'United Kingdom',
    'United States of America',
    'Australia',
    'Canada',
    'Germany',
    'France',
    'South Africa',
    'Brazil',
]


def write_un_excel(path: str, rng: np.random.Generator) -> None:
  """Writes a fallback UN WPP 2024 Excel file if not already downloaded."""
  if os.path.exists(path):
    return
  rows: list[dict[str, Any]] = []
  for country in ['WORLD'] + SAMPLE_COUNTRIES:
    for year in range(2015, 2026):
      row: dict[str, Any] = {
          'Region, subregion, country or area *': country,
          'Year': year,
      }
      for bracket in AGE_BRACKETS:
        row[bracket] = float(rng.integers(500, 5000))
      rows.append(row)
  df = pd.DataFrame(rows)
  with pd.ExcelWriter(path) as writer:
    df.to_excel(writer, index=False, startrow=16)


def generate_results_grid(
    countries: list[str], rng: np.random.Generator
) -> pd.DataFrame:
  """Creates a synthetic 4D MultiIndex results_grid DataFrame."""
  records = []
  idx_tuples = []
  for country in countries:
    for year in range(2015, 2026):
      for gender in [0, 1]:
        for midage in MIDAGES:
          idx_tuples.append((country, year, gender, midage))
          n_readings = int(rng.integers(500, 1500))
          mean_steps = float(rng.normal(7500 - 15 * (midage - 30), 800))
          total_steps = mean_steps * n_readings
          total_steps_sq = (mean_steps**2 + 1500**2) * n_readings
          records.append({
              'steps': total_steps,
              'steps_sq': total_steps_sq,
              'readings': n_readings,
          })
  idx = pd.MultiIndex.from_tuples(
      idx_tuples, names=['country', 'year', 'gender', 'midage']
  )
  return pd.DataFrame(records, index=idx)


def generate_all_dummy_data(data_dir: str) -> None:
  """Generates synthetic internal datasets (and fallback public files if absent)."""
  rng = np.random.default_rng(42)
  internal_dir = os.path.join(data_dir, 'internal')
  public_dir = os.path.join(data_dir, 'public')

  os.makedirs(os.path.join(internal_dir, 'stratify_steps'), exist_ok=True)
  os.makedirs(public_dir, exist_ok=True)

  # 1. Global & Japan checkpoint grids (internal)
  # Precomputed MD5 checkpoint hashes correspond to:
  # - a36731bea99396ebe673f464d73ada56: Global 4D grid (target_country=None,
  #   min_age=20, max_age=104, min_days_per_user=1, max_steps=24*60*60*2)
  # - 9f4397029d8039e43903bb5b8b7e75ec: Japan-only 4D grid (target_country=
  #   'Japan', start_month=11, stop_month=11, exclude_days_of_week=[6],
  #   min_steps=100, max_steps=50000)
  ckpt_dir = os.path.join(internal_dir, 'stratify_steps')
  for ckpt_hash, countries in [
      ('a36731bea99396ebe673f464d73ada56', SAMPLE_COUNTRIES),
      ('9f4397029d8039e43903bb5b8b7e75ec', ['Japan']),
  ]:
    suffix = f'2015-01-01_2026-01-01_{ckpt_hash}.pkl'
    grid_df = generate_results_grid(countries, rng)
    with open(os.path.join(ckpt_dir, f'checkpoint_grid_{suffix}'), 'wb') as f:
      pkl.dump(grid_df, f)
    with open(
        os.path.join(ckpt_dir, f'checkpoint_completed_{suffix}'), 'wb'
    ) as f:
      pkl.dump(True, f)
    with open(
        os.path.join(ckpt_dir, f'checkpoint_processed_{suffix}'), 'wb'
    ) as f:
      pkl.dump(set(range(100)), f)

  # 2. UK 50+ participant-level dataset (2021-2023) (internal)
  uk_rows = []
  for uid in range(1, 201):
    age = int(rng.integers(50, 85))
    midage_uk = 55 if age < 60 else 65 if age < 70 else 75 if age < 80 else 85
    uk_rows.append({
        'user_id': uid,
        'gender': 'male' if uid % 2 == 0 else 'female',
        'birth_year': 2022 - age,
        'age': age,
        'midage_uk': midage_uk,
        'median_daily_steps': float(rng.normal(6500, 1500)),
        'mean_daily_steps': float(rng.normal(6600, 1500)),
        'valid_days': int(rng.integers(15, 100)),
        'year': 2022,
    })
  with open(
      os.path.join(internal_dir, '2021_2023_uk_50plus_participant_steps.pkl'),
      'wb',
  ) as f:
    pkl.dump(pd.DataFrame(uk_rows), f)

  # 3. Public reference files (written only as fallbacks if not already downloaded)
  write_un_excel(
      os.path.join(
          public_dir,
          'WPP2024_POP_F02_2_POPULATION_5-YEAR_AGE_GROUPS_MALE.xlsx',
      ),
      rng,
  )
  write_un_excel(
      os.path.join(
          public_dir,
          'WPP2024_POP_F02_3_POPULATION_5-YEAR_AGE_GROUPS_FEMALE.xlsx',
      ),
      rng,
  )

  who_path = os.path.join(public_dir, 'who_trends.csv')
  if not os.path.exists(who_path):
    who_rows = []
    for country in SAMPLE_COUNTRIES:
      who_rows.append({
          'Country / Territory': country,
          'Men: 2010 Prevalence (95% UI)': f'{rng.uniform(20, 35):.1f}%',
          'Men: 2022 Prevalence (95% UI)': f'{rng.uniform(22, 38):.1f}%',
          'Women: 2010 Prevalence (95% UI)': f'{rng.uniform(25, 40):.1f}%',
          'Women: 2022 Prevalence (95% UI)': f'{rng.uniform(27, 42):.1f}%',
      })
    pd.DataFrame(who_rows).to_csv(who_path, index=False)

  jp_path = os.path.join(public_dir, '2015_2024_japan_steps.pkl')
  if not os.path.exists(jp_path):
    jp_rows = []
    for year in [2015, 2016, 2017, 2018, 2019, 2022, 2023, 2024]:
      for gender in ['male', 'female', 'all']:
        for age_group in [
            'all',
            '20-29',
            '30-39',
            '40-49',
            '50-59',
            '60-69',
            '70+',
        ]:
          jp_rows.append({
              'year': year,
              'gender': gender,
              'age_group': age_group,
              'n_participants': int(rng.integers(200, 1500)),
              'mean_steps': float(rng.normal(7200, 800)),
              'std_steps': float(rng.normal(3200, 300)),
          })
    with open(jp_path, 'wb') as f:
      pkl.dump(pd.DataFrame(jp_rows), f)

  print(
      f'Successfully generated synthetic datasets in:\n  - {internal_dir}\n  -'
      f' {public_dir} (preserved any existing public reference files)'
  )


generate_all_dummy_data(DATA_DIR)